In [4]:
%pip install spacy
!python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.1/12.8 MB 919.0 kB/s eta 0:00:14
      --------------------------------------- 0.2/12.8 MB 1.7 MB/s eta 0:00:08
     - -------------------------------------- 0.6/12.8 MB 3.3 MB/s eta 0:00:04
     ---- ----------------------------------- 1.4/12.8 MB 6.2 MB/s eta 0:00:02
     -------- ------------------------------- 2.8/12.8 MB 10.7 MB/s eta 0:00:01
     -------------- ------------------------- 4.7/12.8 MB 15.0 MB/s eta 0:00:01
     ----------------------- ---------------- 7.5/12.8 MB 20.8 MB/s eta 0:00:01
     --------------------------------- ----- 10.9/12.8 MB 43.7 MB/s eta 0:00:

In [5]:
import pandas as pd
import nltk
import spacy
import re
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download NLTK resources
nltk.download('vader_lexicon')

# Load Spacy model for POS and Dependency Parsing
nlp = spacy.load("en_core_web_sm")

# Initialize VADER Sentiment Analyzer
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\anjan\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [6]:
# Load the primary dataset
df = pd.read_csv('Complaints.csv')

# Remove accidental duplicate headers nested in data
df = df[df['complaint_date'] != 'complaint_date'].copy()

# Load Customer History metadata (Orders)
orders_df = pd.read_csv('Orders.csv')

# Calculate Cumulative Customer History features
df['complaint_date'] = pd.to_datetime(df['complaint_date'], errors='coerce')
df = df.sort_values(by=['customer_id', 'complaint_date']).reset_index(drop=True)
df['prior_complaints'] = df.groupby('customer_id').cumcount()

# Track their total lifetime orders
order_counts = orders_df.groupby('customer_id').size().to_dict()
df['total_orders'] = df['customer_id'].map(order_counts).fillna(0)

# Calculate Ratio (How often do they complain per order?)
df['complaint_ratio'] = (df['prior_complaints'] + 1) / df['total_orders'].apply(lambda x: 1 if x == 0 else x)

### 1. Linguistic Analyzers (Issue Type, Severity, Sentiment, Urgency)

In [7]:
def get_issue_type(text):
    text = str(text).lower()
    if any(word in text for word in ['battery', 'charge', 'charging', 'drain']):
        return 'Battery issue'
    elif any(word in text for word in ['screen', 'display', 'monitor', 'blank', 'flicker', 'lines', 'ghost']):
        return 'Screen damage'
    elif any(word in text for word in ['dead', 'stopped working', 'not turn', 'not power', 'not working', 'stuck', 'restarting', 'boot']):
        return 'Device not working'
    elif any(word in text for word in ['heat', 'hot', 'overheat', 'burnt', 'burn', 'warm']):
        return 'Heating issue'
    elif any(word in text for word in ['lag', 'slow', 'hang', 'freeze', 'performance', 'delay']):
        return 'Performance issue'
    return 'Other'

def get_issue_severity(text):
    text = str(text).lower()
    if any(word in text for word in ['broken', 'dead', 'exploded', 'burnt', 'completely']):
        return 'High'
    elif any(word in text for word in ['restarting', 'unstable', 'disconnect', 'loose']):
        return 'Medium'
    elif any(word in text for word in ['slow', 'minor', 'slight', 'laggy', 'flicker']):
        return 'Low'
    return 'Low'  # Default if not explicitly matched

def get_sentiment(text):
    # VADER returns a dict like {'neg': 0.0, 'neu': 0.323, 'pos': 0.677, 'compound': 0.6369}
    scores = sia.polarity_scores(str(text))
    compound = scores['compound']
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

def get_urgency(text):
    text = str(text)
    # 1. Regex check for multiple punctuation marks indicating urgency
    if re.search(r'!!!|\?\?\?', text):
        return 'High'
    
    # 2. Spacy POS & Dependency Parsing
    doc = nlp(text)
    for token in doc:
        if token.lower_ in ['asap', 'urgent', 'immediately', 'now', 'fast', 'quick']:
            # Check if it is being used as an adverbial modifier, adjectival modifier, or as a root word
            if token.dep_ in ['advmod', 'amod', 'ROOT', 'npadvmod', 'acomp']:
                return 'High'
    
    return 'Normal'

### 2. Rule-Based Fraud Detection (Replacing ML Labels with Full Customer History Logic)

In [8]:
def compute_fraud_features(row):
    text = str(row['complaint_text']).lower()
    score = 6
    reason = "Valid defect"
    classification = "Legitimate"
    refund = "Yes"
    
    # ---- NLP Baseline Rules ----
    # 1. Switcheroo Fraud (Wrong items in box)
    if 'box' in text and any(w in text for w in ['random', 'broken', 'scratches', 'tampered', 'older', 'different']):
        score, classification, refund, reason = 95, 'Fraud', 'No', 'Switcheroo suspected'
        
    # 2. Contradiction Fraud (Claiming wrong color visually)
    elif 'ordered' in text and ('got' in text or 'looks' in text or 'why' in text):
        score, classification, refund, reason = 93, 'Fraud', 'No', 'Color contradiction'

    # 3. Usage Abuse (Using item then returning)
    elif 'used' in text and ('already' in text or 'returning' in text or 'few' in text or 'event' in text or 'now' in text):
        score, classification, refund, reason = 85, 'Fraud', 'No', 'Usage abuse'

    # 4. Unverified Mismatch
    elif any(phrase in text for phrase in ['wrong item', 'wrong model', 'different from', 'not matching', 'match description']): 
        score, classification, refund, reason = 90, 'Fraud', 'No', 'Unverified mismatch'
        
    # 5. Change of Mind (Invalid refund reasons)
    elif any(phrase in text for phrase in ['changed my mind', 'dont like', "don't like", 'prefer', 'not my style', 'expected better', 'not what i expected', 'feels cheap', 'not worth']):
        score, classification, refund, reason = 18, 'Fraud', 'No', 'Change of mind'

    # 6. Vague Complaints (Not describing a valid reason)
    else:
        words = text.split()
        if text.strip() in ['ok', 'bad product', 'not good', 'worst', 'bad', 'meh'] or (len(words) <= 3 and 'dead' not in text and 'not working' not in text):
            score, classification, refund, reason = 22, 'Fraud', 'No', 'Vague complaint'

    # ---- Customer History Penalties ----
    history_penalty = 0
    prior_complaints = row['prior_complaints']
    ratio = row['complaint_ratio']
    
    if prior_complaints >= 3:
        history_penalty += 15
        classification = "Fraud"
        refund = "No"
        reason = "Frequent Abuser"
        
    if ratio > 0.5 and row['total_orders'] > 2:
        history_penalty += 10
        if classification == "Legitimate":
            classification = "Fraud"
            refund = "No"
            reason = "High Ratio Offender"

    if ratio <= 0.1 and prior_complaints == 0:
        history_penalty -= 5  # Loyal/Good Customer discount
        
    score += history_penalty
    
    # Ensure boundaries
    score = max(0, min(100, score))
    if score >= 80:
        classification = "Fraud"
        refund = "No"
        
    return pd.Series([score, classification, refund, reason])

In [9]:
# Apply logic to the dataset
df['Issue_Type'] = df['complaint_text'].apply(get_issue_type)
df['Issue_Severity'] = df['complaint_text'].apply(get_issue_severity)
df['Sentiment'] = df['complaint_text'].apply(get_sentiment)
df['Urgency'] = df['complaint_text'].apply(get_urgency)

# Apply our new explicit rule-based fraud detection across ROWS
df[['Computed_Fraud_Score', 'Computed_Fraud_Class', 'Computed_Refund_Applicable', 'Computed_Fraud_Reason']] = df.apply(compute_fraud_features, axis=1)

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

Defaulting to user installation because normal site-packages is not writeable
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [10]:
# Preview the Results
display(df[['customer_id', 'prior_complaints', 'complaint_text', 'Computed_Fraud_Score', 'Computed_Fraud_Class', 'Computed_Fraud_Reason']].head(10))

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\anjan\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [11]:
# Clean up helper history columns and save
df = df.drop(columns=['prior_complaints', 'total_orders', 'complaint_ratio'])
df.to_csv('Complaints_Extracted.csv', index=False)
print("Extraction complete. Saved to Complaints_Extracted.csv")